# NumPy基础

### 先说为什么用 ndarray

Python 自带的 list 能存东西，但拿它算数很慢。一是 list 里可以混装任意类型，每个元素都是独立的 Python 对象，取出来还要拆箱；二是 list 没有整体运算，想让每个元素乘 2 只能写 for 循环。

NumPy 的 `ndarray` 就是来解决这两件事的：

| | list | ndarray |
|---|---|---|
| 元素类型 | 可以混装 | 必须统一（同一个 `dtype`） |
| 内存 | 存的是对象指针，散在各处 | 一整块连续内存 |
| 运算 | 只能整体复制 / 拼接 | 支持逐元素运算（向量化） |
| 速度 | 慢 | 快一到两个数量级 |

直接比一下 100 万个数求和，感受一下差距。

In [1]:
import time
import numpy as np

n = 1_000_000
lst = list(range(n))
arr = np.arange(n)

t0 = time.time()
s1 = sum(lst)
t1 = time.time()
s2 = arr.sum()
t2 = time.time()

print(s1, s2)
print(f"list    求和: {(t1 - t0) * 1000:.1f} ms")
print(f"ndarray 求和: {(t2 - t1) * 1000:.1f} ms")

499999500000 499999500000
list    求和: 4.1 ms
ndarray 求和: 0.5 ms


每次跑的绝对时间不太一样，但量级差是稳定的。后面写模型时能用数组就不用 for 循环，原因就在这。

### ndarray 的几个特点

- **同质**：所有元素共用同一个 `dtype`，所以 `np.array([1, 2.5])` 会把 1 也变成 `1.`，不会报错，但可能悄悄丢精度。
- **定长**：创建后不能 append。改形状用 `reshape`（只是换一种方式解释同一块内存），真要加元素用 `np.concatenate` 造新数组。
- **多维**：一维是向量，二维是矩阵，三维可以放图像 / 一个 batch，再多就是张量。
- **默认逐元素**：`*` 是逐元素乘，矩阵乘法要单独用 `@`，这点和数学记号不一样，很容易写错。

## 1. 创建数组

常用的就那么几个：

| 函数 | 作用 | 例子 |
|---|---|---|
| `np.array(list)` | 从列表 / 嵌套列表创建 | `np.array([[1, 2], [3, 4]])` |
| `np.arange(start, stop, step)` | 按步长生成，左闭右开 | `np.arange(0, 10, 2)` |
| `np.linspace(start, stop, num)` | 按个数生成，首尾都包含 | `np.linspace(0, 1, 5)` |
| `np.zeros(shape)` | 全 0 | `np.zeros((2, 3))` |
| `np.ones(shape)` | 全 1 | `np.ones(4)` |
| `np.full(shape, value)` | 填同一个值 | `np.full((2, 2), 7)` |
| `np.eye(n)` | 单位矩阵 | `np.eye(3)` |
| `np.random.xxx` | 随机数组 | `np.random.rand(3)` |

In [2]:
a = np.array([1, 2, 3])
b = np.array([[1, 2, 3], [4, 5, 6]])
print(a, a.dtype)
print(b)

print(np.arange(5))
print(np.arange(2, 10, 2))
print(np.arange(0, 1, 0.25))

print(np.linspace(0, 1, 5))

print(np.zeros((2, 3)))
print(np.ones(4))
print(np.full((2, 2), 7))
print(np.eye(3))

[1 2 3] int64
[[1 2 3]
 [4 5 6]]
[0 1 2 3 4]
[2 4 6 8]
[0.   0.25 0.5  0.75]
[0.   0.25 0.5  0.75 1.  ]
[[0. 0. 0.]
 [0. 0. 0.]]
[1. 1. 1. 1.]
[[7 7]
 [7 7]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


`arange` 和 `linspace` 我一开始总分不清，后来这么记：

- `arange(start, stop, step)` 给步长，个数由它算，不含 stop；
- `linspace(start, stop, num)` 给个数，步长由它算，含 stop。

画图、做实验网格的时候一般用 `linspace`，因为点的个数是确定的；下标那种整数序列才用 `arange`。小数步长的 `arange` 会累积浮点误差，最后一个值经常不干净：

In [3]:
print(np.arange(0, 3, 0.1)[-1])
print(np.linspace(0, 3, 31)[-1])

2.9000000000000004
3.0


#### dtype

`dtype` 决定每个元素占几个字节、这块内存怎么解释。常用的几种：

| dtype | 说明 | 典型来源 |
|---|---|---|
| `int64` | 默认整数 | `np.array([1, 2])` |
| `float64` | 默认浮点 | `np.array([1.0, 2.5])` |
| `bool` | 布尔 | 比较运算的结果 |
| `object` | 任意 Python 对象 | 混装时被迫降级，速度慢，尽量别用 |

用 `astype` 转类型，注意**浮点转整数是直接截断**，不是四舍五入。

In [4]:
x = np.array([1.9, -1.9, 2.4])
print(x.dtype)
print(x.astype(int))    # 截断，不是四舍五入
print(np.round(x))

print(np.array([1, 2, 3], dtype=float))
print(np.zeros(3, dtype=int))

float64
[ 1 -1  2]
[ 2. -2.  2.]
[1. 2. 3.]
[0 0 0]


## 2. 数组的属性

| 属性 | 含义 |
|---|---|
| `a.shape` | 各维长度，元组 |
| `a.ndim` | 几维 |
| `a.size` | 元素总数 |
| `a.dtype` | 元素类型 |
| `a.itemsize` | 每个元素多少字节 |
| `a.nbytes` | 整个数组占多少字节 |
| `a.T` | 转置 |

In [5]:
a = np.array([[1, 2, 3], [4, 5, 6]])

print(a.shape)
print(a.ndim)
print(a.size)
print(a.dtype)
print(a.itemsize)
print(a.nbytes)
print(a.T.shape)

(2, 3)
2
6
int64
8
48
(3, 2)


注意一维数组的 shape 是 `(n,)`，不是 `(n, 1)`。它不是一行也不是一列，就是一维，所以转置之后还是自己。后面做广播的时候，这个区别会直接决定代码对不对。

In [6]:
v = np.array([1, 2, 3])
print(v.shape)
print(v.T.shape)
print(v.reshape(3, 1).shape)
print(v.reshape(1, 3).shape)

(3,)
(3,)
(3, 1)
(1, 3)


## 3. 索引和切片

规则和 list 差不多：从 0 开始、可以负号、切片左闭右开。多维数组用逗号把各维分开，写成 `a[行, 列]`，比 `a[行][列]` 快，也清楚。

In [7]:
a = np.arange(12).reshape(3, 4)
print(a)

print(a[0])
print(a[0, 0])
print(a[:, 1])
print(a[0:2, 1:3])
print(a[:, ::2])
print(a[::-1, ::-1])

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[0 1 2 3]
0
[1 5 9]
[[1 2]
 [5 6]]
[[ 0  2]
 [ 4  6]
 [ 8 10]]
[[11 10  9  8]
 [ 7  6  5  4]
 [ 3  2  1  0]]


`a[0]` 拿到的是**一行**，`a[:, 0]` 才是**一列**，这是我最早写错的几个地方之一。另外 `a[:, 0]` 的结果是 (3,) 一维数组，不是 (3, 1) 的列向量，想要列向量得写 `a[:, 0:1]`。

#### 布尔索引

最常用的一种取数方式：把条件写进方括号，留下所有满足条件的元素。条件本身是一个和原数组同形状的 bool 数组。

多个条件不能写 `and` / `or`，要用 `&`、`|`，而且**每个条件都得加括号**（`&` 的优先级比比较运算符高）。

In [8]:
a = np.array([3, 8, 1, 9, 5, 7])

print(a > 5)
print(a[a > 5])
print(a[(a > 3) & (a < 8)])   # 两个条件都要括号

print(np.where(a > 5, 1, 0))
print(np.where(a > 5)[0])

[False  True False  True False  True]
[8 9 7]
[5 7]
[0 1 0 1 0 1]
[1 3 5]


#### 花式索引

用整数列表 / 数组取，一次拿任意几个位置。二维里 `a[行数组, 列数组]` 是**配对**的，不是取子矩阵。

In [9]:
a = np.arange(12).reshape(3, 4)

print(a[[0, 2]])
print(a[[0, 1, 2], [1, 2, 3]])   # 配对取，不是子矩阵
print(a[[0, 2]][:, [1, 3]])

[[ 0  1  2  3]
 [ 8  9 10 11]]
[ 1  6 11]
[[ 1  3]
 [ 9 11]]


#### 视图和副本

这是 NumPy 和 list 差别最大的地方：**基本切片返回视图（view）**，不是副本，改视图等于改原数组。花式索引、布尔索引返回的才是副本。

判断办法：`b.base is a` 为 `True` 就说明 b 是 a 的视图。

In [10]:
a = np.arange(10)

b = a[2:5]        # 视图，改它会改到 a
b[0] = 100
print(a)

c = a[[2, 3]]     # 副本
c[0] = -1
print(a[2])

print(b.base is a)
print(c.base is a)

[  0   1 100   3   4   5   6   7   8   9]
100
True
False


> 做数据预处理时这个坑很费时间：切片取出来的子集一旦被改，原始数据也跟着变，而且不报错，只能靠看结果发现。不放心的切片一律 `a[2:5].copy()`。

## 4. 形状操作

| 操作 | 作用 |
|---|---|
| `a.reshape(...)` | 改形状，元素顺序不变 |
| `a.ravel()` / `a.flatten()` | 拉平成一维（前者尽量返回视图，后者返回副本） |
| `a.T` / `a.transpose()` | 转置 |
| `np.expand_dims(a, axis)` / `a[:, None]` | 加一个长度为 1 的维度 |
| `np.squeeze(a)` | 去掉所有长度为 1 的维度 |
| `np.concatenate` / `vstack` / `hstack` | 拼接 |
| `np.split` / `vsplit` / `hsplit` | 拆分 |

In [11]:
a = np.arange(12)

print(a.reshape(3, 4))
print(a.reshape(2, -1).shape)

b = np.arange(6).reshape(2, 3)
print(b.ravel())
print(b.flatten())
print(b.T)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(2, 6)
[0 1 2 3 4 5]
[0 1 2 3 4 5]
[[0 3]
 [1 4]
 [2 5]]


`reshape` 只是换个方式解释同一块内存，所以元素顺序不变，内存连续时返回视图。`-1` 只能出现一次，多了它算不出来。

`ravel()` 和 `flatten()` 平时写法一样，区别是前者能用视图就不复制，后者永远复制。数据大的时候我就用 `ravel()`。

In [12]:
v = np.array([1, 2, 3])
print(v[:, np.newaxis].shape)
print(v[np.newaxis, :].shape)
print(np.expand_dims(v, axis=0).shape)
print(np.squeeze(np.zeros((1, 3, 1))).shape)

a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6]])
print(np.concatenate([a, b], axis=0))
print(np.vstack([a, b]))
print(np.hstack([a, a]))

c = np.arange(6).reshape(2, 3)
print(np.hsplit(c, 3))
print(np.vsplit(c, 2))

(3, 1)
(1, 3)
(1, 3)
(3,)
[[1 2]
 [3 4]
 [5 6]]
[[1 2]
 [3 4]
 [5 6]]
[[1 2 1 2]
 [3 4 3 4]]
[array([[0],
       [3]]), array([[1],
       [4]]), array([[2],
       [5]])]
[array([[0, 1, 2]]), array([[3, 4, 5]])]


`np.newaxis` 其实就是 `None`，`v[:, None]` 写法一样。形状对不上的时候先别急着改逻辑，多半是差了一个维度。

## 5. 运算和广播

`+ - * / **` 全是**逐元素**的，形状一样就对应位置算。矩阵乘法是单独的 `@`（或 `np.dot`），**不要**拿 `*` 当矩阵乘，这是新手最常见的错。

In [13]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print(a + b)
print(a * b)
print(a ** 2)
print(a @ b)

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print(A * B)
print(A @ B)

[5 7 9]
[ 4 10 18]
[1 4 9]
32
[[ 5 12]
 [21 32]]
[[19 22]
 [43 50]]


#### 广播

形状不一样有时也能直接算，NumPy 会自动把小的那一边拉伸。规则三条：

1. 从**最后一维**开始往前对齐；
2. 对上的每一维，要么长度相等，要么其中一个为 1；
3. 都满足就沿长度为 1 的方向复制（不真的占内存），不满足直接报错。

所以 (3, 1) 和 (4,) 能算，结果 (3, 4)；(3,) 和 (4,) 不行。

In [14]:
a = np.array([[1], [2], [3]])
b = np.array([10, 20, 30, 40])
print((a + b).shape)
print(a + b)

(3, 4)
[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]


广播最实用的场景是每个样本减自己的均值，不用写循环：

In [15]:
x = np.arange(12).reshape(3, 4)

row_mean = x.mean(axis=1)
print(x - row_mean[:, np.newaxis])

print(x - x.mean(axis=1, keepdims=True))

[[-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]]
[[-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]]


#### 聚合和 axis

`sum / mean / std / max / min / argmax` 这类叫聚合，不给 axis 就把整个数组压成一个数。给了 axis 就只压掉那一维：

- `axis=0`：把行压掉，结果是**每一列**的统计量，长度等于列数；
- `axis=1`：把列压掉，结果是**每一行**的统计量，长度等于行数。

我记的是 axis=k 就把第 k 维消掉，比沿行还是沿列好想。

In [16]:
x = np.arange(12).reshape(3, 4)
print(x)

print(x.sum())
print(x.sum(axis=0))    # 每列
print(x.sum(axis=1))    # 每行
print(x.mean(axis=0))
print(x.max())
print(x.argmax())
print(np.argmax(x, axis=0))

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
66
[12 15 18 21]
[ 6 22 38]
[4. 5. 6. 7.]
11
11
[2 2 2 2]


`argmax` 给的是下标不是值，分类任务里把概率转成类别靠的就是它。

顺便记一下最常用的标准化写法，后面处理特征直接套：

In [17]:
X = np.random.rand(5, 3) * 10
X_norm = (X - X.mean(axis=0)) / X.std(axis=0)

print(np.abs(X_norm.mean(axis=0)).max())
print(np.abs(X_norm.std(axis=0) - 1).max())

8.881784197001253e-17
0.0


## 6. 通用函数 ufunc

`np.sin / exp / log / sqrt / abs / maximum` 这些叫 ufunc，特点是**逐元素**、循环写在 C 里，比自己写 for 快得多。Python 自带的 `math.exp` 一次只能吃一个数，`np.exp` 直接吃整个数组。

In [18]:
x = np.array([0, 1, 2, 3])

print(np.exp(x))
print(np.sqrt(x))
print(np.log(np.array([1, np.e, np.e ** 2])))
print(np.abs(np.array([-1, 2, -3])))
print(np.maximum(x, 2))

[ 1.          2.71828183  7.3890561  20.08553692]
[0.         1.         1.41421356 1.73205081]
[0. 1. 2.]
[1 2 3]
[2 2 2 3]


两个要注意的：`np.log(0)` 不报错，给 `-inf`；除数为 0 给 `inf` 再算下去就是 `nan`。它们不会中断程序，但会把一整列数污染掉，所以做除法 / log 之前先看一眼分母。

`np.where` 和 `np.clip` 也很常用：

In [19]:
x = np.array([-2, -1, 0, 1, 2])

print(np.where(x > 0, x, 0))
print(np.clip(x, -1, 1))
print(np.round(np.array([1.234, 5.678]), 1))

[0 0 0 1 2]
[-1 -1  0  1  1]
[1.2 5.7]


`np.where(条件, a, b)` 能当简单的分段函数用，不过它两边都会完整算一遍，只是按条件挑，不是真的短路。

排序相关的：

In [20]:
a = np.array([3, 1, 4, 1, 5])

print(np.sort(a))
print(a)
a.sort()          # 原地排，返回 None
print(a)

print(np.argsort([3, 1, 4, 1, 5]))
print(np.unique([3, 1, 4, 1, 5]))

[1 1 3 4 5]
[3 1 4 1 5]
[1 1 3 4 5]
[1 3 0 2 4]
[1 3 4 5]


`a = a.sort()` 的结果是 `None`，这个错我犯过。`sort` 是原地排序返回 `None`，`sorted()` 才是返回新数组，两个混着用很容易接错。

## 7. 随机数

复现实验结果全靠它。固定的 seed 让每次生成的随机数一模一样，调参和 debug 都靠这个。

In [21]:
np.random.seed(42)

print(np.random.rand(3))
print(np.random.randn(3))
print(np.random.randint(1, 7, size=5))
print(np.random.normal(loc=0, scale=1, size=5))
print(np.random.choice([1, 2, 3], size=3, replace=False))

[0.37454012 0.95071431 0.73199394]
[-1.11188012  0.31890218  0.27904129]
[3 6 5 2 4]
[ 1.01051528  0.02222183 -0.42779291 -0.53181741 -0.1174755 ]
[1 2 3]


`seed` 设一次就行，后面的序列就固定了。但它改的是全局状态，写在库函数里会影响调用方，所以自己写工具函数时我尽量不用。

NumPy 1.17 之后多了个新接口 `default_rng`，不依赖全局状态，现在文档更推荐这个：

In [22]:
rng = np.random.default_rng(42)
print(rng.normal(size=3))
print(rng.integers(0, 10, size=3))

[ 0.30471708 -1.03998411  0.7504512 ]
[0 6 2]


#### 划分训练集和测试集

思路就是打乱下标再切，`sklearn` 的 `train_test_split` 内部干的也是这个事。

In [23]:
idx = np.arange(10)
rng.shuffle(idx)
print(idx)

train = idx[:8]
test = idx[8:]
print(train, test)

[6 8 4 2 9 3 0 7 1 5]
[6 8 4 2 9 3 0 7] [1 5]


## 8. 存和读

训练跑很久的时候，把中间结果存下来比重新算划算。`.npy` 是 NumPy 自己的二进制格式，比 csv 小也快，但只能用 NumPy（或者支持它的库）读。

In [24]:
import os
import tempfile

x = np.arange(12).reshape(3, 4)
tmp = tempfile.mkdtemp()   # 临时目录，别把仓库弄脏

np.save(os.path.join(tmp, "arr.npy"), x)
print(np.load(os.path.join(tmp, "arr.npy")))

np.savetxt(os.path.join(tmp, "arr.csv"), x, delimiter=",", fmt="%d")
print(np.loadtxt(os.path.join(tmp, "arr.csv"), delimiter=","))

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]]


## 9. ML 里的几个式子

前面这些运算拼起来就是模型。先看几个后面会反复用到的：

**线性回归的预测和 MSE**

In [25]:
X = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)
w = np.array([0.5, -1.0])
b = 0.3

y_pred = X @ w + b
print(y_pred)

y_true = np.array([0.5, 1.0, 2.0])
print(((y_pred - y_true) ** 2).mean())

[-1.2 -2.2 -3.2]
13.39


**距离和相似度**（KNN、聚类里天天用）

In [26]:
a = np.array([1, 2, 3])
b = np.array([4, 6, 8])
print(np.sqrt(((a - b) ** 2).sum()))

X = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)
q = np.array([2, 3])
print(np.sqrt(((X - q) ** 2).sum(axis=1)))

7.0710678118654755
[1.41421356 1.41421356 4.24264069]


**sigmoid 和 softmax**

In [27]:
z = np.array([-2.0, 0.0, 2.0])
print(1 / (1 + np.exp(-z)))

scores = np.array([2.0, 1.0, 0.1])
e = np.exp(scores - scores.max())   # 减最大值防溢出
print(e / e.sum())

[0.11920292 0.5        0.88079708]
[0.65900114 0.24243297 0.09856589]
